# Práctica 4-1A: Segmentación de Señales de Tránsito con YOLOv11
**Universidad Politécnica Salesiana — Visión por Computador**  
**Período Lectivo:** Marzo – Agosto 2026  
**Docente:** Ing. Vladimir Robles Bykbaev  
**Estudiante:** Jordy Romero

## Dataset
- **Stop sign / Señal de pare**: COCO 2017 val (imágenes reales con máscaras de segmentación poligonal)
- **Traffic light / Semáforo**: COCO 2017 val (imágenes reales con máscaras de segmentación poligonal)
- **Traffic cone / Cono**: Imágenes sintéticas generadas con OpenCV

| ID | Clase | Fuente | Imágenes train |
|----|-------|--------|----------------|
| 0 | stop-sign | COCO 2017 val | ~48 |
| 1 | traffic-light | COCO 2017 val | ~129 |
| 2 | traffic-cone | Sintético OpenCV | 60 |

## 1. Verificar entorno

In [1]:
import subprocess, torch, os, glob, time, uuid, shutil, random
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import psutil
from pathlib import Path
from IPython.display import Image as IPyImage, display

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
mac = ':'.join(['{:02x}'.format((uuid.getnode()>>i)&0xff) for i in range(0,48,8)][::-1])
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA:        {torch.cuda.is_available()}')
print(f'GPU:         {torch.cuda.get_device_name(0)}')
print(f'VRAM:        {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')
print(f'MAC Address: {mac}')

Tue Jul  7 20:08:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   34C    P0             34W /  132W |       2MiB /   8192MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Dataset (ya preparado)

In [2]:
DATASET = 'dataset_traffic'
yaml_path = os.path.join(DATASET, 'data.yaml')

print('=== DATASET ===')
for split in ['train', 'valid', 'test']:
    imgs = glob.glob(f'{DATASET}/{split}/images/*')
    lbls = glob.glob(f'{DATASET}/{split}/labels/*.txt')
    print(f'{split:6s}: {len(imgs):3d} imágenes, {len(lbls):3d} etiquetas')

class_counts = {0:0, 1:0, 2:0}
for lbl in glob.glob(f'{DATASET}/train/labels/*.txt'):
    for line in open(lbl):
        c = int(line.split()[0])
        if c in class_counts: class_counts[c] += 1

print(f'\nAnnotations en train:')
print(f'  [0] stop-sign:     {class_counts[0]}')
print(f'  [1] traffic-light: {class_counts[1]}')
print(f'  [2] traffic-cone:  {class_counts[2]}')
print(f'\ndata.yaml: {os.path.abspath(yaml_path)}')

=== DATASET ===
train : 237 imágenes, 237 etiquetas
valid :  53 imágenes,  53 etiquetas
test  :  56 imágenes,  56 etiquetas

Annotations en train:
  [0] stop-sign:     52
  [1] traffic-light: 448
  [2] traffic-cone:  103

data.yaml: /home/jordy/Documents/Vision-Artificial/Practica4-1/Parte1A/dataset_traffic/data.yaml


## 3. Visualización de muestras

In [ ]:
os.makedirs('resultados', exist_ok=True)
train_images = glob.glob(f'{DATASET}/train/images/*.jpg')[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Muestras del Dataset — Señales de Tránsito', fontsize=13, fontweight='bold')

for ax, img_path in zip(axes.flatten(), train_images):
    img = cv2.imread(img_path)
    if img is None: continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path)[:25], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('resultados/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.close()
display(IPyImage('resultados/dataset_samples.png'))
print('Muestras visualizadas -> resultados/dataset_samples.png')

## 4. Fine-Tuning YOLOv11-seg (Transfer Learning)

In [ ]:
from ultralytics import YOLO

model = YOLO('models/yolo11n-seg.pt')
print(f'YOLOv11n-seg cargado | Dispositivo: {"GPU" if torch.cuda.is_available() else "CPU"}')
print(f'Parámetros: {sum(p.numel() for p in model.model.parameters())/1e6:.1f}M')

t0 = time.time()
results = model.train(
    data=yaml_path,
    epochs=60,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='runs',
    name='traffic_signs_v1',
    exist_ok=True,
    patience=15,
    save=True,
    plots=True,
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    augment=True,
    degrees=10.0,
    fliplr=0.5,
    mosaic=1.0,
    verbose=False,
)
# Copiar best.pt al directorio models/ para acceso directo
_candidates = glob.glob('runs/**/best.pt', recursive=True)
if _candidates:
    shutil.copy2(sorted(_candidates)[-1], 'models/best.pt')
print(f'\nEntrenamiento completado en {(time.time()-t0)/60:.1f} minutos')
print(f'Modelo disponible en: models/best.pt')

## 5. Validación y Métricas

In [ ]:
import glob as _glob, os as _os

# Usar models/best.pt si existe, si no buscar en runs/
MODEL_PATH = 'models/best.pt' if _os.path.exists('models/best.pt') else sorted(_glob.glob('runs/**/best.pt', recursive=True))[-1]
print(f'Modelo: {MODEL_PATH}')

best_model = YOLO(MODEL_PATH)
val_results = best_model.val(data=yaml_path, imgsz=640, verbose=False)

print('=== MÉTRICAS DE VALIDACIÓN ===')
print(f'mAP50 (Box):     {val_results.box.map50:.4f}')
print(f'mAP50-95 (Box):  {val_results.box.map:.4f}')
print(f'mAP50 (Mask):    {val_results.seg.map50:.4f}')
print(f'mAP50-95 (Mask): {val_results.seg.map:.4f}')
print(f'Precisión:       {val_results.box.mp:.4f}')
print(f'Recall:          {val_results.box.mr:.4f}')

train_dir = _os.path.dirname(_os.path.dirname(MODEL_PATH))
for plot in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    p = f'{train_dir}/{plot}'
    if _os.path.exists(p):
        print(f'\n{plot}'); display(IPyImage(p))

## 6. Inferencia sobre imágenes de test

In [ ]:
test_images = glob.glob(f'{DATASET}/test/images/*.jpg')[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Inferencia YOLOv11-seg — Señales de Tránsito', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), test_images):
    res = best_model(img_path, conf=0.25, verbose=False)
    img_result = cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(img_result)
    ax.set_title(os.path.basename(img_path)[:25], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('resultados/inferencia_test.png', dpi=150, bbox_inches='tight')
plt.close()
display(IPyImage('resultados/inferencia_test.png'))
print('Inferencia completada -> resultados/inferencia_test.png')

## 7. Benchmark GPU vs CPU

In [ ]:
def benchmark(model_path, images, device, n=50):
    m = YOLO(model_path)
    for img in images[:3]:
        m(img, device=device, verbose=False)
    times, ram = [], []
    for i in range(n):
        t0 = time.perf_counter()
        m(images[i % len(images)], device=device, verbose=False)
        times.append(time.perf_counter() - t0)
        ram.append(psutil.virtual_memory().used / 1024**3)
    return {'fps': 1/np.mean(times), 'ms': np.mean(times)*1000, 'ram': np.mean(ram)}

test_imgs = glob.glob(f'{DATASET}/test/images/*.jpg')[:10]

print('Benchmark CPU (50 inferencias)...')
cpu = benchmark(MODEL_PATH, test_imgs, 'cpu')
print('Benchmark GPU (50 inferencias)...')
gpu = benchmark(MODEL_PATH, test_imgs, 0)

vram_used  = torch.cuda.memory_allocated()/1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory/1024**3

print(f"\n{'='*54}")
print(f"    COMPARACIÓN GPU vs CPU — YOLOv11-seg")
print(f"    GPU: {torch.cuda.get_device_name(0)}")
print(f"{'='*54}")
print(f"{'Métrica':<26} {'CPU':>12} {'GPU':>12}")
print(f"{'-'*54}")
print(f"{'FPS':<26} {cpu['fps']:>12.1f} {gpu['fps']:>12.1f}")
print(f"{'Tiempo/img (ms)':<26} {cpu['ms']:>12.1f} {gpu['ms']:>12.1f}")
print(f"{'RAM (GB)':<26} {cpu['ram']:>12.2f} {gpu['ram']:>12.2f}")
print(f"{'VRAM usada (GB)':<26} {'N/A':>12} {vram_used:>12.2f}")
print(f"{'VRAM total (GB)':<26} {'N/A':>12} {vram_total:>12.2f}")
print(f"{'-'*54}")
print(f"{'Aceleración GPU/CPU':<26} {gpu['fps']/cpu['fps']:>12.1f}x")
print(f"{'='*54}")

# nvidia-smi
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(f'\n{r.stdout}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Comparación GPU vs CPU — YOLOv11-seg\nDataset: Señales de Tránsito', fontsize=13, fontweight='bold')

for ax, (vals, title, unit) in zip(axes, [
    ([cpu['fps'], gpu['fps']], 'FPS (frames/segundo)', 'FPS'),
    ([cpu['ms'],  gpu['ms']],  'Tiempo inferencia (ms/img)', 'ms')
]):
    bars = ax.bar(['CPU','GPU'], vals, color=['#E74C3C','#2ECC71'], edgecolor='black', width=0.4)
    ax.set_title(title, fontsize=12); ax.set_ylabel(unit)
    ax.set_ylim(0, max(vals)*1.3); ax.grid(axis='y', alpha=0.3)
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
                f'{v:.1f}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('resultados/benchmark_gpu_vs_cpu.png', dpi=150, bbox_inches='tight')
plt.close()
display(IPyImage('resultados/benchmark_gpu_vs_cpu.png'))
print('Gráfica guardada -> resultados/benchmark_gpu_vs_cpu.png')

## 8. Inferencia en tiempo real (Webcam)

In [ ]:
def run_webcam(model_path, device=0, cam=0):
    m = YOLO(model_path)
    cap = cv2.VideoCapture(cam)
    if not cap.isOpened():
        print('Error: No se pudo abrir la cámara'); return
    device_name = 'GPU' if device == 0 else 'CPU'
    print(f'{device_name} — presiona Q para salir')
    fps_list = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        t0 = time.perf_counter()
        res = m(frame, device=device, conf=0.35, verbose=False)
        fps = 1/(time.perf_counter()-t0)
        fps_list.append(fps)
        out = res[0].plot()
        color = (0,255,0) if device==0 else (0,0,255)
        cv2.putText(out, f'{device_name} | FPS: {fps:.1f}', (10,35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
        cv2.putText(out, 'YOLOv11-seg | Traffic Signs', (10,70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
        cv2.imshow(f'YOLOv11-seg [{device_name}]', out)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release(); cv2.destroyAllWindows()
    if fps_list: print(f'FPS promedio: {np.mean(fps_list):.1f}')

# Descomenta la línea según el dispositivo:
# run_webcam(MODEL_PATH, device=0)      # GPU
# run_webcam(MODEL_PATH, device='cpu')  # CPU
print('Función lista — descomenta una línea para ejecutar con webcam')

## 9. Exportar modelo

In [ ]:
best_model.export(format='onnx', imgsz=640)
print('Modelo exportado a ONNX')
print('\nPráctica 4-1A completada')
print('Clases: stop-sign | traffic-light | traffic-cone')